# MCS603 — Programming II (Python)
## Week 9: Exception Handling

---
### Learning Objectives
- Understand Python's exception model and the exception hierarchy
- Use `try`, `except`, `else`, and `finally` blocks correctly
- Catch specific exceptions and multiple exception types
- Raise exceptions with `raise` and create custom exception classes
- Apply exception handling to real-world file I/O and user input

### Outline
1. What Is an Exception?  
2. The `try / except` Block  
3. Catching Specific Exceptions  
4. `else` and `finally` Clauses  
5. Raising Exceptions  
6. Custom Exception Classes  
7. Exception Chaining  
8. Common Exception Types  
9. Lab Exercises  
10. Assignment  

---
## 1. What Is an Exception?

An **exception** is an event that disrupts normal program flow at runtime.

Without handling, exceptions terminate the program with a traceback:

```
ZeroDivisionError: division by zero
FileNotFoundError: [Errno 2] No such file or directory: 'data.txt'
ValueError: invalid literal for int() with base 10: 'hello'
```

### Python Exception Hierarchy (partial)
```
BaseException
 └── Exception
      ├── ArithmeticError
      │    ├── ZeroDivisionError
      │    └── OverflowError
      ├── LookupError
      │    ├── IndexError
      │    └── KeyError
      ├── OSError (IOError)
      │    └── FileNotFoundError
      ├── TypeError
      ├── ValueError
      ├── AttributeError
      └── RuntimeError
```

---
## 2. The `try / except` Block

```python
try:
    # code that might raise an exception
except ExceptionType:
    # code that runs if ExceptionType is raised
```

In [ ]:
# Without handling
# result = 10 / 0   # ZeroDivisionError — crashes program

# With handling
try:
    result = 10 / 0
except ZeroDivisionError:
    print("Cannot divide by zero!")
    result = None

print("Program continues. result =", result)

In [ ]:
# Safe integer input
def safe_int(s):
    try:
        return int(s)
    except ValueError:
        print(f"'{s}' is not a valid integer.")
        return None

print(safe_int("42"))     # 42
print(safe_int("3.14"))   # error message, None
print(safe_int("hello"))  # error message, None

---
## 3. Catching Specific Exceptions

Always catch the **most specific** exception you can — catching bare `Exception` hides bugs.

You can handle multiple exception types in one `except` clause or with separate clauses.

In [ ]:
def divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        print("Error: division by zero")
    except TypeError:
        print(f"Error: cannot divide {type(a).__name__} by {type(b).__name__}")
    return None

print(divide(10, 2))      # 5.0
print(divide(10, 0))      # zero division
print(divide("a", 2))     # type error

In [ ]:
# Capture the exception object with 'as'
try:
    data = [1, 2, 3]
    print(data[10])    # IndexError
except (IndexError, KeyError) as e:
    print(f"Lookup error: {type(e).__name__}: {e}")

# Access exception details
try:
    int("not a number")
except ValueError as e:
    print(f"ValueError caught: {e}")
    print(f"Args: {e.args}")

---
## 4. `else` and `finally` Clauses

```python
try:
    risky_operation()
except SomeError:
    handle_error()
else:
    # runs ONLY if NO exception was raised in try
    success_action()
finally:
    # ALWAYS runs — exception or not
    cleanup()
```

| Clause | Runs when |
|---|---|
| `except` | An exception was raised |
| `else` | No exception was raised |
| `finally` | Always — even after `return` or unhandled exception |

In [ ]:
def read_score(filename):
    try:
        f = open(filename, "r")
        score = int(f.readline().strip())
    except FileNotFoundError:
        print(f"File '{filename}' not found.")
        score = None
    except ValueError:
        print("File does not contain a valid integer.")
        score = None
    else:
        print(f"Score read successfully: {score}")
    finally:
        print("(Attempted file read — done)")
        try:
            f.close()
        except:
            pass
    return score

# Write a test file
with open("score.txt", "w") as f:
    f.write("87\n")

print("--- Valid file ---")
read_score("score.txt")

print("\n--- Missing file ---")
read_score("missing.txt")

---
## 5. Raising Exceptions

Use `raise` to signal that an error condition has occurred in your own code.

In [ ]:
def set_age(age):
    if not isinstance(age, int):
        raise TypeError(f"Age must be int, got {type(age).__name__}")
    if age < 0 or age > 150:
        raise ValueError(f"Age {age} is out of valid range (0–150)")
    return age

# Valid
print(set_age(25))

# Invalid — will be caught
for bad in ["twenty", -1, 200]:
    try:
        set_age(bad)
    except (TypeError, ValueError) as e:
        print(f"{type(e).__name__}: {e}")

In [ ]:
# Re-raise — handle partially then let it propagate
def load_config(path):
    try:
        with open(path) as f:
            return f.read()
    except FileNotFoundError:
        print(f"Config file '{path}' missing — using defaults.")
        raise   # re-raises the same FileNotFoundError

try:
    load_config("config.json")
except FileNotFoundError:
    print("Caller also caught it.")

---
## 6. Custom Exception Classes

Create domain-specific exceptions by subclassing `Exception`.

In [ ]:
# Custom exception hierarchy for a student system
class StudentError(Exception):
    """Base class for all student-related errors."""

class InvalidGradeError(StudentError):
    def __init__(self, grade, valid_range=(0, 100)):
        self.grade = grade
        super().__init__(f"Grade {grade} is outside valid range {valid_range}")

class StudentNotFoundError(StudentError):
    def __init__(self, student_id):
        super().__init__(f"No student with ID {student_id} found")


def record_grade(student_id, grade, db):
    if student_id not in db:
        raise StudentNotFoundError(student_id)
    if not (0 <= grade <= 100):
        raise InvalidGradeError(grade)
    db[student_id]["grade"] = grade

database = {2024001: {"name": "Alice"}, 2024002: {"name": "Bob"}}

tests = [(2024001, 85), (9999999, 70), (2024002, 110)]
for sid, grade in tests:
    try:
        record_grade(sid, grade, database)
        print(f"Recorded grade {grade} for student {sid}")
    except StudentError as e:
        print(f"StudentError: {e}")

---
## 7. Common Exception Types Quick Reference

| Exception | Common Cause |
|---|---|
| `ValueError` | Wrong value type (`int("abc")`) |
| `TypeError` | Wrong data type (`"a" + 1`) |
| `ZeroDivisionError` | Division or modulo by zero |
| `IndexError` | List index out of range |
| `KeyError` | Dict key not found |
| `AttributeError` | Object has no such attribute |
| `FileNotFoundError` | File/directory missing |
| `PermissionError` | OS permission denied |
| `ImportError` | Module not found |
| `OverflowError` | Numeric result too large |
| `RecursionError` | Maximum recursion depth exceeded |

In [ ]:
examples = [
    ("ValueError",      lambda: int("abc")),
    ("TypeError",       lambda: "a" + 1),
    ("ZeroDivision",    lambda: 1 / 0),
    ("IndexError",      lambda: [][0]),
    ("KeyError",        lambda: {}["x"]),
    ("AttributeError",  lambda: (42).upper()),
    ("FileNotFound",    lambda: open("ghost.txt")),
]

for name, fn in examples:
    try:
        fn()
    except Exception as e:
        print(f"{name:<18} → {type(e).__name__}: {e}")

---
## 8. Lab Exercises

### Lab 1: Safe Input Loop
Write a function `get_positive_float(prompt)` that:
- Keeps asking the user until they provide a valid positive float
- Handles `ValueError` for non-numeric input
- Handles the case where the number ≤ 0 by raising a custom `NegativeInputError`

In [ ]:
class NegativeInputError(ValueError):
    pass

def get_positive_float(prompt):
    # TODO
    pass

# Test with simulated input
test_inputs = ["abc", "-5", "0", "3.14"]
idx = 0
def mock_input(p):
    global idx
    val = test_inputs[idx]; idx += 1; print(f"  Input: {val}"); return val

import builtins
builtins.input = mock_input
# print(get_positive_float("Enter a positive number: "))

### Lab 2: File Error Handling
Write `safe_read_csv(filepath)` that:
- Returns the contents as a list of dicts
- Handles `FileNotFoundError` (returns empty list)
- Handles `PermissionError` (re-raises with a helpful message)
- Logs the outcome to `operations.log`

In [ ]:
import csv

def safe_read_csv(filepath):
    # TODO
    pass

print(safe_read_csv("grades.csv"))    # should return list
print(safe_read_csv("missing.csv"))   # should return []

### Lab 3: Exception-Hardened Calculator
Build a calculator that handles ALL of the following gracefully:
- Non-numeric input for numbers
- Division by zero
- Invalid operator
- Integer overflow (very large exponentiation)
- Unknown error (catch-all with logging)

In [ ]:
def calculate(a_str, op, b_str):
    # TODO
    pass

test_cases = [
    ("10",  "+", "5"),
    ("10",  "/", "0"),
    ("abc", "+", "5"),
    ("2",  "^", "1000000"),
    ("10",  "@", "5"),
]
for a, op, b in test_cases:
    result = calculate(a, op, b)
    print(f"{a} {op} {b} = {result}")

---
## 9. Assignment — Week 9

**Build a Robust Student Record System**

Extend the file-based contact manager from Week 8 with full exception handling.

**Custom Exceptions (20 pts)**  
Define: `RecordError`, `DuplicateIDError`, `RecordNotFoundError`, `InvalidFieldError`

**Protected Operations (40 pts)**  
Each operation must handle:
- File I/O errors (`FileNotFoundError`, `PermissionError`)
- Data validation errors (ID out of range, score not 0–100, empty name)
- Duplicate student IDs on insert
- Lookup of non-existent IDs on update/delete

**Logging (20 pts)**  
Write a `log_event(level, message)` function that appends to `system.log` with timestamps.  
Use levels: `INFO`, `WARNING`, `ERROR`

**Test Suite (20 pts)**  
Write 10 test calls that deliberately trigger each exception type and verify the program continues running.

In [ ]:
# Implement here


---
## Summary — Week 9

| Concept | Key Point |
|---|---|
| `try / except` | Catch specific exception types, not bare `Exception` |
| `else` | Runs only when no exception occurred |
| `finally` | Always runs — use for cleanup |
| `raise` | Signal errors from your own code |
| Custom exceptions | Subclass `Exception`; add context in `__init__` |
| Exception object | `except Error as e` — access `e.args`, message |

**Next:** Week 10–11 — Web Development with Flask & Django